In [1]:
import os
import random
import json
import asyncio
import nest_asyncio
from datetime import datetime
from openai import AsyncOpenAI
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import logfire
load_dotenv()
logfire.configure(token=os.environ.get("LOGFIRE_TOKEN"), console=False)
logfire.instrument_openai()

In [2]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
timestamp

'20250723_1526'

In [3]:
nest_asyncio.apply()

In [4]:
client = AsyncOpenAI()

In [5]:
semaphore = asyncio.Semaphore(5)

In [6]:
# model="gpt-4o-mini"
# model="o3-mini"
# model="o4-mini"
# model="gpt-4.1"
model="gpt-4o"

In [ ]:
async def generate_questions_for_file(filepath: str, n_questions: int = 2) -> list[dict]:
    """
    Generate N quqesion-answer pairs for a given event file using gpt-4.1
    returns a list of dicts with question, answer and filename
    """
    async with semaphore:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        original_title = lines[1].strip('# ').strip()
        clean_content = "".join(lines[2:])
        filename = os.path.basename(filepath)

        prompt = f"""
        You are a world class expert AI engineer. Your role is to generate an evaluation dataset, consisting of query-answer pairs. Adherence to the outlined rules is the only measure of success.
        
        ### CONTEXT
        **Original Title:** 
        <original_title>"{original_title}"</original_title>

        <event_content>
        {clean_content}
        </event_content>

        **Current Date:** 
        <current_date>{datetime.now().date().isoformat()}</current_date>

        ### RULES
        You must follow each one of these rules in the following order.

        **1. DATE LOGIC (ABSOLUTELY CRUCIAL FIRST STEP):**
        - **Analyse:** Find the event's start date in the context
            - If the event context contains multiple dates, select the one closest to the current_date.
        - **Apply Logic:**
            - **If the event is in the PAST:** The 'question' field in your JSON output **MUST** contain a specific timeframe (e.g., "w na początku czerwca 2025", "w ostatni weekend lipca", "wakacje 2025"), making this particular event identifiable.
                - **Example of a past event (Date: 01.06.2025):**
                    - **INVALID QUESTION** 'piknik rodzinny w Warszawie'; 'piknik rodzinny w czerwcu'
                    - **VALID QUESTION** 'piknik rodzinny w warszawie na początku czerwca'; 'piknik rodzinny w warszawie pierwszego czerwca'
            - **If the event is in the FUTURE:** The 'question' does not require a specific timeframe.

        **2 QUESTION SIMPLIFICATION (Secondary Guideline):**
        - **After** satisfying the date rule, create a simple query.
        - Do no use the original title or niche jargon. Identify the underlying theme of the event, and focus on general terms a person might use.
            - **GOOD:** 'warsztaty japońskiego malarstwa tuszem', 'bieg charytatywny w Warszawie w sierpniu'
            - **BAD:** 'warsztaty Suibokuga', 'bieg charytatywny na rzecz onkologii'
        - **If an event relates to a specific type of activity**, it **MUST** be mentioned in the question.
            - **GOOD:** 'mecz piłkarski w Warszawie w Listopadzie', 'zajęcia pilates w czerwcu na Ursynowie'
            - **BAD:** 'wydarzenie sportowe w Warszawie w Listopadzie', 'zajęcia zdrowotne na Ursynowie'
        - **If an event relates to a specific entity**, the entity **MUST** be mentioned in the question.
            - **GOOD:** 'koncert zespołu Kult w Warszawie w wakacje', 'Mecz Legii w listpadzie'
            - **BAD:** 'koncert w Warszawie w wakacje', 'mecz piłkarski w listopadzie'
        - The question must be simple and straightforward (the question reader will not see the article OR title OR any other information about the event)


        ### OUTPUT FORMAT
        Return ONLY a single, valid JSON object. Do not include any other text, explanations, or markdown formatting around the JSON.
        ```json
        {{
            "questions": [
                {{
                    "question": "The single, rule-compliant user search query.",
                    "answer": "Title: {original_title}\\nDate: [Comprehensive Date/Time Summary]\\nLocation: [Full Location]"
                }}
            ]
        }}
        ```
        """
        response = await client.chat.completions.create(
            model=model,
            messages=[
                # {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.7
            )
        try:
            response_content = response.choices[0].message.content
            if not response_content:
                return []
            response_json = json.loads(response_content)
            if isinstance(response_json, list):
                qa_pairs = response_json
            elif isinstance(response_json, dict):
                qa_pairs = response_json.get("pairs", response_json.get("questions", response_json.get("data", []))) 
                if not isinstance(qa_pairs, list):
                    return []
            else:
                qa_pairs = []
        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            return []
        except Exception as e:
            print(f"Unexpected error: {e}")
            return []

        results = []
        for pair in qa_pairs:
            results.append({
                "question": pair["question"],
                "answer": pair["answer"],
                "filename": filename
            })
        
        return results

async def generate_random_questions(n_pages: int = 5, questions_per_page: int = 2) -> list[dict]:
    """
    Generate random questions from a set of event files
    """
    event_files = [f for f in os.listdir("./data/events") if f.endswith(".md")]
    selected_files = random.sample(event_files, min(n_pages, len(event_files)))
    tasks = []
    for filename in selected_files:
        filepath = os.path.join("./data/events", filename)
        tasks.append(generate_questions_for_file(filepath, questions_per_page))
    all_results = []
    with tqdm(total=len(tasks), desc="Generating questions") as pbar:
        for coro in asyncio.as_completed(tasks):
            try:
                result = await coro
                all_results.append(result)
                pbar.update(1)
            except Exception as e:
                pbar.update(1)
                continue

    all_questions = []
    for questions in all_results:
        all_questions.extend(questions)
    return all_questions

@logfire.instrument("Generate Queries", extract_args=True, record_return=True)
async def main():
    n_pages = 10
    questions = await generate_random_questions(n_pages=n_pages, questions_per_page=1)
    print(f"\nGenerated {len(questions)} total questions")

    for i, q in enumerate(questions): 
        print(f"\n{i+1}. Q: {q['question']}")
        print(f"   A: {q['answer']}")
        print(f"   File: {q['filename']}")
    
    return questions

questions = await main()

Generating questions:   0%|          | 0/10 [00:00<?, ?it/s]


Generated 10 total questions

1. Q: złożenie kwiatów pod tablicą pamięci w Warszawie
   A: Title: Złożenie kwiatów pod tablicą upamiętniającą Krzysztofa Kamila Baczyńskiego
Date: 4 sierpnia 2025, godz. 12:00 - 13:00
Location: ul. Hołówki 3, Warszawa
   File: złożenie_kwiatów_pod_tablicą_upamiętniającą_krzysztofa_kamila_baczyńskiego_00021.md

2. Q: zumba fitness w Warszawie w lipcu
   A: Title: Zumba Fitness z BOSem
Date: 29.07.2025 19:00 - 29.07.2025 20:00
Location: Białołęcki Ośrodek Sportu, Krzyżówki 22, 03-126 Warszawa
   File: zumba_fitness_z_bosem_00305.md

3. Q: zajęcia na basenie dla dzieci w Warszawie w lipcu
   A: Title: Lato w mieście 2025 - wejście indywidualnie Pływalnia "Wodnik" 30.06 – 25.07.2025 r.
Date: 30.06.2025 13:00 - 25.07.2025 15:00
Location: Pływalnia "Wodnik", ul. gen. T. Komorowskiego 40, Warszawa
   File: lato_w_mieście_2025_wejście_indywidualnie_pływalnia_wodnik_3006_25072025_r_00148.md

4. Q: spacer po Warszawie śladami dworców na początku lipca
   A: Title

In [ ]:
model

In [ ]:
filename=f"./synth_datasets/raw_queries/Polish_{timestamp}_{len(questions)}_{model}.json"
filename

In [ ]:
with open(filename, "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2, ensure_ascii=False)